# ek_logreg_or
Logistic regression OR-combination · features: (s_freq, s_env) from z-score pipeline · event F0.5 threshold tuning

In [ ]:
%%script true
from google.colab import drive
drive.mount('/content/drive')

%cd drive/MyDrive/Colab Notebooks/sentinel
!pip install -e .

import sys
sys.path.insert(0, "/content/drive/MyDrive/Colab Notebooks/sentinel/src")

In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import uniform_filter1d
import json

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from sentinel.ml_logic.metrics import compute_all_metrics, esa_metric
from sentinel.ml_logic.thresholds import tune_threshold
from sentinel.ml_logic.data import find_anomaly_segments

ANOMALY_COLOR = '#e74c3c'
NOMINAL_COLOR = '#2980b9'

In [ ]:
# ── config ────────────────────────────────────────────────────────────────────
FREQ_CHANNELS = [
    'channel_41', 'channel_42', 'channel_43',
    'channel_44', 'channel_45', 'channel_46',
]
TRAIN_FRAC     = 0.70
ROLLING_WINDOW = 300

USE_FREQ_CHANNELS = True
USE_ENV_CHANNELS  = True

ENV_CHANNELS  = ['channel_14', 'channel_21', 'channel_29']
ENV_WINDOW    = 200
ENV_MA_WINDOW = 5_000

# ── logistic regression ───────────────────────────────────────────────────────
LR_C        = 1.0    # regularization (smaller = stronger)
LR_MAX_ITER = 1000
N_SWEEP_THR = 200    # threshold sweep resolution

RAW_DIR  = Path('./data/raw')
SUB_DIR  = Path('./submissions')
SUB_DIR.mkdir(exist_ok=True)

In [ ]:
cols  = ['id', 'is_anomaly'] + FREQ_CHANNELS + ENV_CHANNELS
train = pq.read_table(str(RAW_DIR / 'train.parquet'), columns=cols
                      ).to_pandas().set_index('id')
test  = pq.read_table(str(RAW_DIR / 'test.parquet'),
                      columns=['id'] + FREQ_CHANNELS + ENV_CHANNELS).to_pandas().set_index('id')
print(f'Train: {len(train):,} rows  |  Test: {len(test):,} rows')

In [ ]:
n       = len(train)
tr_end  = int(n * TRAIN_FRAC)
val_end = n

y_tr  = train['is_anomaly'].values[:tr_end]
y_val = train['is_anomaly'].values[tr_end:]
print(f'Train: {tr_end:,} | Val: {n - tr_end:,}')
print(f'Val anomaly rate: {y_val.mean():.4%}')

## [Optional] Frequency decomposition
Пропустить → FREQ_CHANNELS остаются сырыми.  
Выполнить → заменяются на `_res` подканалы.

In [ ]:
SUBCHANNEL_PARTS  = ['res']
BIG_WINDOW_DECOMP = 50_000

if not USE_FREQ_CHANNELS:
    print('FREQ channels disabled — skipping decomposition.')
else:
    freq_map = json.loads(Path('./data/freq_map.json').read_text())

    def _decompose(arr, bw, mw, sw):
        a    = np.asarray(arr, dtype=np.float64)
        sin1 = uniform_filter1d(a,          size=bw, mode='nearest')
        r1   = a - sin1
        s2r  = uniform_filter1d(r1,         size=mw, mode='nearest')
        s3r  = uniform_filter1d(r1 - s2r,   size=sw, mode='nearest')
        sin2 = uniform_filter1d(r1 - s3r,   size=mw, mode='nearest')
        r2   = r1 - sin2
        sin3 = uniform_filter1d(r2,         size=sw, mode='nearest')
        return sin1, sin2, sin3, r2 - sin3

    _base_ch = [c for c in FREQ_CHANNELS
                if not c.endswith(('_sin1', '_sin2', '_sin3', '_res'))]
    for df, label in [(train, 'train'), (test, 'test')]:
        print(f'Decomposing {label} ...')
        for ch in _base_ch:
            T1, T2 = freq_map[ch]
            mw = max(T1 * 2, T2 // 3)
            sw = max(3, T1 // 5)
            *_, res = _decompose(df[ch].values, BIG_WINDOW_DECOMP, mw, sw)
            df[ch + '_res'] = res

    FREQ_CHANNELS = [ch + '_res' for ch in _base_ch]
    print(f'FREQ_CHANNELS: {FREQ_CHANNELS}')

In [ ]:
if not USE_ENV_CHANNELS:
    print('ENV channels disabled — skipping.')
else:
    def _envr_col(series, env_w, ma_w):
        env = series.rolling(window=env_w, min_periods=1).min()
        ma  = env.rolling(window=ma_w, min_periods=1, center=True).mean()
        return (env - ma).values

    print(f'Computing envelope residuals  ENV_WINDOW={ENV_WINDOW}  ENV_MA_WINDOW={ENV_MA_WINDOW} ...')
    for df, label in [(train, 'train'), (test, 'test')]:
        for ch in ENV_CHANNELS:
            df[ch + '_envr'] = _envr_col(df[ch], ENV_WINDOW, ENV_MA_WINDOW)
    ENV_CHANNELS = [ch + '_envr' for ch in ENV_CHANNELS]
    print(f'ENV features: {ENV_CHANNELS}')

In [ ]:
_active = []
if USE_FREQ_CHANNELS: _active += FREQ_CHANNELS
if USE_ENV_CHANNELS:  _active += ENV_CHANNELS
assert _active, 'No channels enabled'

CHANNELS = _active
print(f'Active CHANNELS ({len(CHANNELS)}): {CHANNELS}')

In [ ]:
def rolling_zscore_df(df, window):
    roll = df.rolling(window=window, min_periods=max(2, window // 10))
    mu   = roll.mean()
    sd   = roll.std().fillna(1.0).clip(lower=1e-8)
    return ((df - mu) / sd).fillna(0.0)

def top_p_mean(z_arr, p):
    n_top = max(1, int(z_arr.shape[1] * p))
    top   = np.partition(np.abs(z_arr), -n_top, axis=1)[:, -n_top:]
    return top.mean(axis=1)

In [ ]:
print(f'Computing z-scores on {len(train):,} rows ...')

freq_cols = [c for c in CHANNELS if not c.endswith('_envr')]
env_cols  = [c for c in CHANNELS if c.endswith('_envr')]

_parts = []
if freq_cols:
    _parts.append(rolling_zscore_df(train[freq_cols], ROLLING_WINDOW))
if env_cols:
    std_tr_env = train[env_cols].iloc[:tr_end].std()
    _parts.append(train[env_cols] / std_tr_env)

z_raw      = pd.concat(_parts, axis=1)[CHANNELS]
ref_per_ch = z_raw.iloc[:tr_end].abs().quantile(0.99).clip(lower=1e-8)
z_full     = z_raw / ref_per_ch
z_val      = z_full.iloc[tr_end:]
print('Done.')

## Group scores
Isolated k-search per group to find best aggregation within each group.  
Reuses the same logic as ek_baseline_zscore.

In [ ]:
def _find_best_k(z_df, y_true, label):
    n = z_df.shape[1]
    best_sc, best_k_ = -1, 1
    print(f'  k  F0.5  [{label}]')
    for k_ in range(1, n + 1):
        scores = top_p_mean(z_df.values, p=k_ / n)
        res = tune_threshold(scores, y_true, metric_fn=esa_metric,
                             lo_percentile=(0, 90), hi_percentile=(1, 99.99), n_sweep=80)
        print(f'  {k_}  {res["score"]:.4f}')
        if res['score'] > best_sc:
            best_sc, best_k_ = res['score'], k_
    print(f'  -> best k={best_k_}/{n}  F0.5={best_sc:.4f}\n')
    return best_k_ / n

print('-- Isolated k-search -----------------------------------')
best_p_freq = _find_best_k(z_val[freq_cols], y_val, 'FREQ') if freq_cols else 1.0
best_p_env  = _find_best_k(z_val[env_cols],  y_val, 'ENV')  if env_cols  else 1.0

In [ ]:
s_freq_val = top_p_mean(z_val[freq_cols].values, p=best_p_freq) if freq_cols else np.zeros(len(y_val))
s_env_val  = top_p_mean(z_val[env_cols].values,  p=best_p_env)  if env_cols  else np.zeros(len(y_val))

print(f's_freq  mean={s_freq_val.mean():.3f}  std={s_freq_val.std():.3f}  '
      f'p99={np.percentile(s_freq_val, 99):.3f}  max={s_freq_val.max():.3f}')
print(f's_env   mean={s_env_val.mean():.3f}  std={s_env_val.std():.3f}  '
      f'p99={np.percentile(s_env_val, 99):.3f}  max={s_env_val.max():.3f}')

## Logistic Regression
Input: (s_freq, s_env) — 2 features per timestep.  
`class_weight='balanced'` compensates for ~10% anomaly rate.  
Threshold tuned on predicted probability to maximise esa_f05.

In [ ]:
X_val_lr = np.column_stack([s_freq_val, s_env_val])

scaler   = StandardScaler()
X_val_s  = scaler.fit_transform(X_val_lr)

lr = LogisticRegression(
    class_weight='balanced',
    C=LR_C,
    solver='lbfgs',
    max_iter=LR_MAX_ITER,
)
lr.fit(X_val_s, y_val)

w_freq, w_env = lr.coef_[0]
print(f'Coefficients:  w_freq={w_freq:.3f}  w_env={w_env:.3f}  bias={lr.intercept_[0]:.3f}')
print('(both positive = OR-like; larger weight = dominant group)')

In [ ]:
proba_val = lr.predict_proba(X_val_s)[:, 1]

res = tune_threshold(proba_val, y_val, metric_fn=esa_metric,
                     lo_percentile=(0, 90), hi_percentile=(1, 99.99), n_sweep=N_SWEEP_THR)
best_thr_lr = res['threshold']
print(f'Best threshold: {best_thr_lr:.4f}  F0.5={res["score"]:.4f}')

y_pred_val = (proba_val > best_thr_lr).astype(np.int8)

metrics = compute_all_metrics(y_val, y_pred_val)
print('\n-- Validation metrics (LR OR-combination) ---------------')
for k, v in metrics.items():
    print(f'  {k:<28} {v}')

In [ ]:
# OR thresholds per group — for boundary comparison in scatter plot
def _group_thr(scores, y_true, label):
    res = tune_threshold(scores, y_true, metric_fn=esa_metric,
                         lo_percentile=(0, 90), hi_percentile=(1, 99.99), n_sweep=200)
    print(f'{label}: thr={res["threshold"]:.4f}  F0.5={res["score"]:.4f}')
    return res['threshold']

thr_freq_or = _group_thr(s_freq_val, y_val, 'FREQ')
thr_env_or  = _group_thr(s_env_val,  y_val, 'ENV')

In [ ]:
# ── Decision space: s_freq vs s_env ──────────────────────────────────────────
# Subsampled scatter + LR boundary + OR boundary for comparison
step = max(1, len(y_val) // 50_000)
idx  = np.arange(0, len(y_val), step)
sf_s = s_freq_val[idx]
se_s = s_env_val[idx]
y_s  = y_val[idx]

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(sf_s[y_s == 0], se_s[y_s == 0],
           s=0.5, alpha=0.15, color=NOMINAL_COLOR, label='nominal', rasterized=True)
ax.scatter(sf_s[y_s == 1], se_s[y_s == 1],
           s=3, alpha=0.7, color=ANOMALY_COLOR, label='anomaly', rasterized=True)

# LR decision boundary: w_freq*(sf-mu_f)/s_f + w_env*(se-mu_e)/s_e + bias = logit(thr)
w_freq, w_env  = lr.coef_[0]
mu_f, s_f = scaler.mean_[0], scaler.scale_[0]
mu_e, s_e = scaler.mean_[1], scaler.scale_[1]
logit_thr = np.log(best_thr_lr / (1 - best_thr_lr))

sf_range = np.linspace(0, np.percentile(s_freq_val, 99.9), 300)
sf_scaled = (sf_range - mu_f) / s_f
se_scaled = (logit_thr - lr.intercept_[0] - w_freq * sf_scaled) / w_env
se_lr = se_scaled * s_e + mu_e

ax.plot(sf_range, se_lr, 'k-', lw=2, label='LR boundary')

# OR boundary (two thresholds = rectangle corner)
ax.axvline(thr_freq_or, color='seagreen',  ls='--', lw=1.5, label=f'OR thr_freq={thr_freq_or:.2f}')
ax.axhline(thr_env_or,  color='mediumpurple', ls='--', lw=1.5, label=f'OR thr_env={thr_env_or:.2f}')

ax.set_xlim(0, np.percentile(s_freq_val, 99.9))
ax.set_ylim(0, np.percentile(s_env_val,  99.9))
ax.set_xlabel('s_freq')
ax.set_ylabel('s_env')
ax.set_title('Decision space: s_freq vs s_env  |  LR vs OR boundary')
ax.legend(fontsize=9, markerscale=5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Submission

In [ ]:
freq_cols_t = [c for c in CHANNELS if not c.endswith('_envr')]
env_cols_t  = [c for c in CHANNELS if c.endswith('_envr')]

_parts_test = []
if freq_cols_t:
    _parts_test.append(rolling_zscore_df(test[freq_cols_t], ROLLING_WINDOW))
if env_cols_t:
    _parts_test.append(test[env_cols_t] / std_tr_env)

z_test = pd.concat(_parts_test, axis=1)[CHANNELS] / ref_per_ch

s_freq_test = top_p_mean(z_test[freq_cols_t].values, p=best_p_freq) if freq_cols_t else np.zeros(len(test))
s_env_test  = top_p_mean(z_test[env_cols_t].values,  p=best_p_env)  if env_cols_t  else np.zeros(len(test))

X_test_s   = scaler.transform(np.column_stack([s_freq_test, s_env_test]))
proba_test = lr.predict_proba(X_test_s)[:, 1]
y_pred_test = (proba_test > best_thr_lr).astype(int)

sub      = pd.DataFrame({'id': test.index, 'is_anomaly': y_pred_test})
out_path = SUB_DIR / 'ek_logreg_or.csv'
sub.to_csv(out_path, index=False)
print(f'Saved  {out_path}')
print(f'Predicted anomaly rate: {y_pred_test.mean():.4%}')

## Potential improvements

### Variant B — per-channel z-scores (9 features)
Replace `(s_freq, s_env)` with all 9 per-channel z-scores as LR inputs.  
LR learns per-channel weights — `top_p_mean` not needed (tasks 1+2 move to model).  
More flexible, 9 coefficients instead of 2.

### log1p transform
Scores are heavy-tailed (anomalies 10-30x above nominal).  
`log1p(s_freq)`, `log1p(s_env)` compress tails — LR separates classes more cleanly.

### Interaction term
Add `s_freq * s_env` as a third feature.  
Models the case 'both groups active simultaneously'.

### Regularization tuning
Sweep `LR_C` via cross-validation within val.  
With 2 features and 4M rows overfitting is unlikely, but may help generalisation.

### GBM (next level)
Replace LR with LightGBM on the same 2 features.  
Non-linear boundary: high s_freq alone triggers anomaly independently of s_env.  
New groups = new features, pipeline structure unchanged.